# PREDICTION

## EBRB

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from math import sqrt
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import tensorflow as tf
from tensorflow import random as tf_random
from keras.models import Sequential
from keras.layers import LSTM, Dense
from sklearn.metrics import mean_absolute_error
import random

In [7]:
Aw=np.array([1,1,1,1,1,1,1]);#前提属性权重都是1
x = pd.read_csv('train.csv',encoding="gbk")
xtst = pd.read_csv('test.csv',encoding="gbk")

In [8]:
for column in x.columns:
    x[column] = pd.to_numeric(x[column], errors='coerce')      ##转换为数值型数据，避免报错


In [9]:
for column in xtst.columns:
    xtst[column] = pd.to_numeric(xtst[column], errors='coerce')      ##转换为数值型数据，避免报错


In [10]:
D=x.pop('直接经济损失(亿元)')
Dtest=xtst.pop('直接经济损失(亿元)')         ##取出训练集和测试集的结果属性

In [11]:
Dtest=pd.DataFrame(Dtest).reset_index().drop(columns=['index'])              ##转换成df

In [13]:
def equal_partition_df(input_df, n):
    # 初始化一个空的DataFrame，用于存放结果
    result_df = pd.DataFrame()
    
    # 对于输入DataFrame的每一列
    for column in input_df.columns:
        # 获取该列的最小值和最大值
        min_value = input_df[column].min()
        max_value = input_df[column].max()
        
        # 生成等分后的数组
        partitioned_array = np.linspace(min_value, max_value, num=n)
        
        # 将数组转换为DataFrame，并进行纵向拼接
        partitioned_df = pd.DataFrame(partitioned_array, columns=[column])
        result_df = pd.concat([result_df, partitioned_df], axis=1)
        result = result_df.T
    
    return result


df = pd.DataFrame(x)
df2 = pd.DataFrame(D)
urate = 7    #前提属性评价等级数
Durate = 8     #结果属性评价等级数
u = equal_partition_df(df, urate)
Du = equal_partition_df(df2, Durate).T
Du = Du.squeeze(1)
Du=np.array(Du)

In [14]:
##基于效用的转换方法
xnum=x.shape[0]
innum=len(D)
unum=len(Du)
b=np.zeros((innum,unum)) 
for i in range(innum):
     for j in range(unum-1): #unum5
            # print(Du[j],D[i],Du[j+1])
            if Du[j] <= D[i] and D[i] <= Du[j+1]:
                b[i,j]=(Du[j+1]-D[i])/(Du[j+1]-Du[j])
                b[i,j+1]=1-b[i,j]

In [15]:
def EBRB(x, u, Aw, xtst, b):
   #训练数据集计算分布式置信度
    innuml = x.shape[1] #获取矩阵x的列数
    innumh = x.shape[0] #获取矩阵x的行数
    unum = u.shape[1]  #获取矩阵u的列数

    A = np.zeros((innumh,unum,innuml))#构造三维零矩阵
    for k in range(0,innumh):
        for i in range(0,innuml):
            for j in range(0,unum-1):
                if u.iloc[i,j] <= x.iloc[k,i] and x.iloc[k,i] <= u.iloc[i,j+1]:
                    A[k,j,i] = (u.iloc[i,j+1]-x.iloc[k,i]) / (u.iloc[i,j+1]-u.iloc[i,j])
                    A[k,j+1,i] = 1 - A[k,j,i]
    

                    
    #测试数据集计算分布式置信度
    innumltst = xtst.shape[1] #列数
    innumhtst = xtst.shape[0] #行数
    unum = u.shape[1] #获取矩阵u的列数(keshenglue)

    Atst = np.zeros((innumhtst, unum, innumltst))
    for k in range(0,innumhtst):
        for j in range(0,unum-1):
            for i in range(0,innumltst):
                if u.iloc[i,j] <= xtst.iloc[k,i] and xtst.iloc[k,i] <= u.iloc[i,j+1]:
                    Atst[k,j,i] = (u.iloc[i,j+1]-xtst.iloc[k,i]) / (u.iloc[i,j+1]-u.iloc[i,j])
                    Atst[k,j+1,i] = 1 - Atst[k,j,i]

    #计算个体匹配度
    E = []
    weight=[]
    L = A.shape[0] # 规则数(矩阵x)
    L1 = Atst.shape[0]  # 输入数据x数量，即xtst
    M = Aw.shape[0]  # 前提属性个数
    N = x.shape[1]  # 评价等级数
    Aw1 = Aw / np.max(Aw) 
    dist1=[]
    dist=[]
    sk=[]
    sk3=[]
    sk1=[]
    sk1b=[]
    for i1 in range(0,L1):
        for j1 in range(0,L):
            x2 = Atst[i1]
            a2 = A[j1]
            dist1.append(x2 - a2) 
    for i in range(len(dist1)):
        dist.append(np.sum(dist1[i]**2,axis=0))
    for i in range(len(dist)):
        sk.append(1-(np.divide(dist[i],2)**0.5))
    
    
    #计算激活权重
    count=0
    for j in range(L1):
        sk2=[]
        for i in range(L):
            sk2.append(sk[count*L+i])
        count=count+1
        sk3.append(sk2)
    for i in range(L1):
        for j in range(L):
            sk4=sk3[i][j]
            Aw1_repeated = np.tile(Aw1, int(np.ceil(len(sk4) / len(Aw1))))[:len(sk4)]
            sk1b.append(np.power(sk4, Aw1_repeated))
        sk1.append(sk1b)
        sk1b=[]
    for i in range(L1):
        E1=[]
        for j in range(L):
            E1.append(np.prod(sk1[i][j]))
        E.append(E1)

    weight = [[value/row_sum for value in row] for row, row_sum in zip(E, np.sum(E,axis=1))]

    # 合成激活规则
    F=[]
    F1=[]
    E1=[]
    k1=[]
    mi=[]
    mh1=[]   
    for i2 in range(L1):
        n = b.shape[1] # 评价等级数
        F.append(np.repeat(weight[i2],n)) 
    F1=np.array(F).reshape(L1,L,n)*np.array(b)
    mnj = F1.transpose([0, 2, 1])
    mhj1= np.negative(weight) + 1
    B = -np.sum(b,axis=1) + 1#B = -np.sum(b) + 1
    C = np.array(B).T#B.T
    mhj2= (np.array(weight))* C
    D1 = np.zeros((L1, L, n))
    for i2 in range(0,L1):
        for i in range(0,n):
            D1[i2,:,i] = mnj[i2][i] + mhj1[i2] + mhj2[i2]
        E1.append( mhj1[i2,:] + mhj2[i2,:])
    for i2 in range(0,L1):
        k1.append( 1 / (np.sum(np.prod(D1[i2], axis=0)) - (n-1) * np.prod(E1[i2])))
        mi.append( k1[i2] * (np.prod(D1[i2], axis=0) - np.prod(E1[i2])))
        mh1.append( k1[i2] * np.prod(mhj1[i2]))
    bi=[]
    for i in range(L1):
        bi.append(mi[i] / (1 - mh1[i]))
        
    return pd.DataFrame(bi)
########

In [16]:
bi=EBRB(x, u, Aw, xtst, b)
f=[]
l=bi.shape[0]
m=bi.shape[1]
print(l)
print(m)


133
8


In [17]:

for i in range(l):
    sum=bi.iloc[i].sum()
    bN=bi.iloc[i][m-1]
    b1=bi.iloc[i][0]
    f.append((b1+bN)/2*(1-sum)+np.dot(bi.iloc[i],Du))
f = pd.DataFrame(f,columns=['prediction'])
f = f.round(6)
bi_rounded = bi.round(6)
result = pd.concat([bi_rounded, Dtest,f],axis=1)
output = result.rename(columns={0:'D1',1:"D2",2:"D3",3:"D4",4:"D5",5:"D6",6:"D7",7:"D8"})
output.index = range(1, len(output) + 1)

In [18]:
output  #输出表格

,D1,D2,D3,D4,D5,D6,D7,D8,直接经济损失(亿元),prediction
1,0.999556,0.000444,0.000000,0.0,0.0,0.0,0.000000,0.000000,2.312470e-05,0.000010
2,0.999084,0.000913,0.000000,0.0,0.0,0.0,0.000000,0.000003,6.204780e-06,0.000020
3,0.998103,0.001896,0.000000,0.0,0.0,0.0,0.000000,0.000000,1.038550e-04,0.000040
4,0.989974,0.010024,0.000001,0.0,0.0,0.0,0.000000,0.000001,4.478757e-03,0.000210
5,0.993738,0.006260,0.000002,0.0,0.0,0.0,0.000000,0.000000,1.941992e-03,0.000131
...,...,...,...,...,...,...,...,...,...,...
129,0.990441,0.009403,0.000003,0.0,0.0,0.0,0.000000,0.000153,5.422610e-05,0.000219
130,0.998657,0.001341,0.000000,0.0,0.0,0.0,0.000000,0.000002,9.041220e-06,0.000029
131,0.997498,0.002498,0.000000,0.0,0.0,0.0,0.000000,0.000004,3.598690e-07,0.000053
132,0.998698,0.001271,0.000000,0.0,0.0,0.0,0.000000,0.000031,4.072370e-07,0.000031


In [20]:
mse = mean_squared_error(output['直接经济损失(亿元)'], output['prediction'])
rmse = sqrt(mse)
# 计算平均绝对误差 (MAE)
mae = sm.tools.eval_measures.meanabs(output['直接经济损失(亿元)'], output['prediction'])
# 计算平均百分比误差 (MAPE)
mape = sm.tools.eval_measures.meanabs(output['直接经济损失(亿元)'] - output['prediction'], output['直接经济损失(亿元)'])
# 打印结果
print("均方根误差 (RMSE):", rmse)
print("平均绝对误差 (MAE):", mae)


均方根误差 (RMSE): 0.0010957400680418614
平均绝对误差 (MAE): 0.00043352796539172935
